### <font color="greenblue"> Ponderação rake: Raking of replicate weight design - Iterative Proportional Fitting (IPF) </font>

In [1]:
import numpy as np
import pandas as pd

def rake_weights(df, weight_col, margins, control=None):
    """
    Raking (IPF) por margens 1D.
    
    df: DataFrame com colunas das variáveis de margem
    weight_col: nome da coluna de peso a ajustar
    margins: dict {var: {categoria: total_pop}}
             ex.: {"sexo": {"M": 5200, "F": 4800}, "idade": {"18-34": 3000, ...}}
    control: dict com "maxit" e "epsilon"
    """
    if control is None:
        control = {"maxit": 100, "epsilon": 1e-10}

    w = df[weight_col].to_numpy(dtype=float)

    # Checagens básicas (missing em variáveis de margem quebra o algoritmo)
    for var in margins.keys():
        if df[var].isna().any():
            raise ValueError(f"Há missing na variável de margem '{var}'. Trate antes de rakear.")

    for it in range(control["maxit"]):
        w_old = w.copy()
        max_rel_change = 0.0

        # Ajusta uma margem por vez
        for var, pop_totals in margins.items():
            # Totais atuais por categoria (ponderados)
            current = (
                pd.DataFrame({var: df[var].values, "_w": w})
                .groupby(var)["_w"].sum()
                .to_dict()
            )

            # Aplica fator categoria-a-categoria
            for cat, pop_total in pop_totals.items():
                cur_total = current.get(cat, 0.0)

                # Se a categoria não aparece na amostra, não dá para ajustar (divisão por zero)
                if cur_total <= 0:
                    raise ValueError(
                        f"Categoria '{cat}' (var='{var}') tem total atual 0 na amostra. "
                        f"Rake não consegue ajustar: revise categorias/margens/amostra."
                    )

                ratio = pop_total / cur_total
                mask = (df[var].values == cat)
                w[mask] *= ratio

        # Critério de convergência: maior mudança relativa de peso
        rel_change = np.max(np.abs(w - w_old) / (np.abs(w_old) + 1e-12))
        max_rel_change = max(max_rel_change, rel_change)

        if max_rel_change < control["epsilon"]:
            break

    return w

def make_bootstrap_replicates(df, base_weight_col, R=30, seed=123):
    """
    Cria replicate weights por bootstrap simples:
    - reamostra índices com reposição
    - replicate weight = base_weight * contagem_de_vezes_que_o_i_aparece
    """
    rng = np.random.default_rng(seed)
    n = len(df)
    base_w = df[base_weight_col].to_numpy(dtype=float)

    Wrep = np.zeros((n, R), dtype=float)
    for r in range(R):
        sample_idx = rng.integers(0, n, size=n)  # n draws with replacement
        counts = np.bincount(sample_idx, minlength=n)
        Wrep[:, r] = base_w * counts
    return Wrep

def check_margins(df, w, margins):
    """
    Retorna um dict com totais ponderados por categoria, para checar se bate com a população.
    """
    out = {}
    for var, pop_totals in margins.items():
        tot = (
            pd.DataFrame({var: df[var].values, "_w": w})
            .groupby(var)["_w"].sum()
            .reindex(pop_totals.keys())
        )
        out[var] = pd.DataFrame({
            "weighted_total": tot.values,
            "population_total": [pop_totals[k] for k in pop_totals.keys()],
        }, index=list(pop_totals.keys()))
    return out

In [2]:
# -----------------------------
# EXEMPLO TOY (você pode trocar pelos seus dados)
# -----------------------------
df = pd.DataFrame({
    "sexo": ["M","F","F","M","F","M","M","F","M","F"],
    "idade": ["18-34","18-34","35-54","35-54","55+","55+","18-34","35-54","55+","18-34"],
})

# Peso inicial (ex.: todos 1 só para simplificar)
df["w_base"] = 1.0
df

,sexo,idade,w_base
0,M,18-34,1.0
1,F,18-34,1.0
2,F,35-54,1.0
3,M,35-54,1.0
4,F,55+,1.0
5,M,55+,1.0
6,M,18-34,1.0
7,F,35-54,1.0
8,M,55+,1.0
9,F,18-34,1.0


In [3]:
path = r"C:\Users\rayner.santos\Downloads\Consistencia e Processamento\Cielo\EMP_Cielo Satisfacao_3onda_NOV25_2025.12.26.xlsx"
data = pd.read_excel(path, sheet_name="BD_CODIGOS")

In [4]:
df = data[["ID_EMP", "ONDA", "TRAB_C", "PF_PJ", "SEG_NOVO_BU", "VREG", "POND"]].copy()
df = df.loc[(df["TRAB_C"] == 1) & (df["ONDA"] == 59)]
df["w_base"] = 1.0
df

,ID_EMP,ONDA,TRAB_C,PF_PJ,SEG_NOVO_BU,VREG,POND,w_base
0,3000230,59,1,2,2,2,0.999119,1.0
2,3000383,59,1,2,1,2,0.317763,1.0
9,3001301,59,1,1,1,2,0.470412,1.0
10,3000014,59,1,2,2,3,0.994190,1.0
13,3000716,59,1,2,2,1,1.340074,1.0
...,...,...,...,...,...,...,...,...
15351,3004051,59,1,2,3,3,1.167923,1.0
15352,3004052,59,1,2,3,4,0.811759,1.0
15353,3004053,59,1,2,3,2,1.183012,1.0
15354,3004054,59,1,2,3,3,1.167923,1.0


In [5]:
for col in ["PF_PJ", "SEG_NOVO_BU", "VREG"]:
    print(df[col].value_counts(dropna=False), "\n")

PF_PJ
2    1719
1     197
Name: count, dtype: int64 

SEG_NOVO_BU
2    1068
1     514
3     334
Name: count, dtype: int64 

VREG
1    466
2    454
3    325
4    311
6    246
5    114
Name: count, dtype: int64 



In [27]:
df_margins = pd.DataFrame(
    {
        "Colunas": ["PF_PJ", "PF_PJ", "SEG_NOVO_BU", "SEG_NOVO_BU", "SEG_NOVO_BU", "VREG", "VREG", "VREG", "VREG", "VREG", "VREG"],
        "Codigo": [1, 2, 1, 2, 3, 1, 2, 3, 4, 5, 6],
        "Label": ["PF", "PJ", "Empreendedores", "Varejo", "GC", "São Paulo", "Sudeste (Sem São Paulo)", "Sul", "Centro-Oeste", "Norte", "Nordeste"],
        "Total marginais": [214, 1702, 277, 1275, 364, 531, 373, 309, 209, 123, 371]
    }
)
display(df_margins)

# df_margins -> margins (dict de dict)
margins = (
    df_margins.dropna(subset=["Total marginais"])
        .groupby("Colunas")
        .apply(lambda g: dict(zip(g["Codigo"], g["Total marginais"])), include_groups=False)
        .to_dict()
)
print("\n", margins)

,Colunas,Codigo,Label,Total marginais
0,PF_PJ,1,PF,214
1,PF_PJ,2,PJ,1702
2,SEG_NOVO_BU,1,Empreendedores,277
3,SEG_NOVO_BU,2,Varejo,1275
4,SEG_NOVO_BU,3,GC,364
5,VREG,1,São Paulo,531
6,VREG,2,Sudeste (Sem São Paulo),373
7,VREG,3,Sul,309
8,VREG,4,Centro-Oeste,209
9,VREG,5,Norte,123



 {'PF_PJ': {1: 214, 2: 1702}, 'SEG_NOVO_BU': {1: 277, 2: 1275, 3: 364}, 'VREG': {1: 531, 2: 373, 3: 309, 4: 209, 5: 123, 6: 371}}


In [ ]:
# Margens populacionais (totais)
# margins = {
#     "sexo": {"M": 5200, "F": 4800},
#     "idade": {"18-34": 3000, "35-54": 4000, "55+": 3000}
# }

margins = {
    "PF_PJ": {1: 214, 2: 1702},
    "SEG_NOVO_BU": {1: 277, 2: 1275, 3: 364},
    "VREG": {1: 531, 2: 373, 3: 309, 4: 209, 5: 123, 6: 371}
}
margins

{'PF_PJ': {1: 214, 2: 1702},
 'SEG_NOVO_BU': {1: 277, 2: 1275, 3: 364},
 'VREG': {1: 531, 2: 373, 3: 309, 4: 209, 5: 123, 6: 371}}

In [29]:
# 1) Rake no peso principal
w_raked = rake_weights(df, "w_base", margins, control={"maxit": 5000, "epsilon": 1e-10})
df["w_raked"] = w_raked
display(df)

print("Checagem das margens (peso principal raked):")
checks = check_margins(df, df["w_raked"].values, margins)
for var, tab in checks.items():
    print("\n", var)
    print(tab)

,ID_EMP,ONDA,TRAB_C,PF_PJ,SEG_NOVO_BU,VREG,POND,w_base,w_raked
0,3000230,59,1,2,2,2,0.999119,1.010489,1.010489
2,3000383,59,1,2,1,2,0.317763,0.380962,0.380962
9,3001301,59,1,1,1,2,0.470412,0.583839,0.583839
10,3000014,59,1,2,2,3,0.994190,1.112214,1.112214
13,3000716,59,1,2,2,1,1.340074,1.347854,1.347854
...,...,...,...,...,...,...,...,...,...
15351,3004051,59,1,2,3,3,1.167923,0.973521,0.973521
15352,3004052,59,1,2,3,4,0.811759,0.676659,0.676659
15353,3004053,59,1,2,3,2,1.183012,0.884481,0.884481
15354,3004054,59,1,2,3,3,1.167923,0.973521,0.973521


Checagem das margens (peso principal raked):

 PF_PJ
   weighted_total  population_total
1           214.0               214
2          1702.0              1702

 SEG_NOVO_BU
   weighted_total  population_total
1           277.0               277
2          1275.0              1275
3           364.0               364

 VREG
   weighted_total  population_total
1           531.0               531
2           373.0               373
3           309.0               309
4           209.0               209
5           123.0               123
6           371.0               371


In [48]:
# 2) Replicate weights (bootstrap) + rake em cada replicate
R = 20
Wrep = make_bootstrap_replicates(df, "w_base", R=R, seed=2026)
print(Wrep_raked)

Wrep_raked = np.zeros_like(Wrep)
for r in range(R):
    df["_wrep"] = Wrep[:, r]
    Wrep_raked[:, r] = rake_weights(df, "_wrep", margins, control={"maxit": 100, "epsilon": 1e-10})
print(Wrep_raked)

[[0.         0.         0.         ... 1.00396729 0.         1.02379807]
 [0.         0.         0.38239816 ... 0.406055   0.77309601 1.10668678]
 [0.54228416 0.         0.         ... 1.72535074 1.28943026 0.        ]
 ...
 [1.81288508 1.86714289 0.86819442 ... 0.84230317 0.86252446 2.56304879]
 [1.99285308 2.79687557 0.         ... 0.         0.90223086 0.        ]
 [0.67124354 0.         1.26969369 ... 0.62488718 0.6046014  1.38907047]]
[[0.         0.         0.         ... 1.00396729 0.         1.02379807]
 [0.         0.         0.38239816 ... 0.406055   0.77309601 1.10668678]
 [0.54228416 0.         0.         ... 1.72535074 1.28943026 0.        ]
 ...
 [1.81288508 1.86714289 0.86819442 ... 0.84230317 0.86252446 2.56304879]
 [1.99285308 2.79687557 0.         ... 0.         0.90223086 0.        ]
 [0.67124354 0.         1.26969369 ... 0.62488718 0.6046014  1.38907047]]


In [10]:
# Exemplo: estimar uma média ponderada de uma variável y e erro padrão via replicates
# Vamos criar y fictício
df["y"] = [10, 12, 14, 11, 9, 13, 8, 15, 7, 16]

def wmean(y, w):
    return np.sum(y * w) / np.sum(w)

theta = wmean(df["y"].values, df["w_raked"].values)
thetas_rep = np.array([wmean(df["y"].values, Wrep_raked[:, r]) for r in range(R)])
se_boot = np.sqrt(np.mean((thetas_rep - theta) ** 2))

print("\nEstimativa (média ponderada) com rake:", theta)
print("Erro padrão (bootstrap replicates raked):", se_boot)


Estimativa (média ponderada) com rake: 11.58350482571519
Erro padrão (bootstrap replicates raked): nan


C:\Users\rayner.santos\AppData\Local\Temp\ipykernel_4904\2477881720.py:6: RuntimeWarning: invalid value encountered in scalar divide
  return np.sum(y * w) / np.sum(w)
